# Plot energy spectra at particular times.

In [ ]:
import glob
import numpy as np
import pandas as pd
import datetime as dt

import matplotlib
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

from JunoPy.jedi import JediData, merge_two_jedi_data, get_jedi_data
from JunoPy.index import IndexFile
from JunoPy.mag import MagData
from JunoPy.library import find_ticks_between

In [ ]:
%%time
t1 = dt.datetime.strptime('2017-076T06:44:50', '%Y-%jT%H:%M:%S')
# t1 = dt.datetime.strptime('2017-077T02:56:16', '%Y-%jT%H:%M:%S')
# t1 = dt.datetime.strptime('2017-075T23:57:09', '%Y-%jT%H:%M:%S')

start = t1 - dt.timedelta(minutes=10)
end = t1 + dt.timedelta(minutes=10)

electron_data, proton_data, nonproton_data = get_jedi_data(start, end)

In [ ]:
    t1 = ['2017-03-17 06:44:30', '2017-03-17 06:45:00']
    t2 = ['2017-03-17 06:41:30', '2017-03-17 06:42:00']

# t1 = ['2017-03-18 02:56:01', '2017-03-18 02:56:31']
# t2 = ['2017-03-18 02:51:01', '2017-03-18 02:51:31']
    
# t1 = ['2017-03-16 23:57:01', '2017-03-16 23:57:31']
# t2 = ['2017-03-16 23:54:01', '2017-03-16 23:54:31']

In [ ]:

def geom_mean(a1, a2, axis=None):
    
    return np.sqrt(a1 * a2)

In [ ]:
def plot_single_data(data, t, ax, **kwargs):
    
    energybins = geom_mean(data.energybins_low, data.energybins_high)
    
    xerror = np.zeros((2, len(energybins)))
    
    xerror[0, :] = energybins - data.energybins_low
    xerror[1, :] = data.energybins_high - energybins

    data_t = data.energytime[t[0]: t[1]].values.T
    
    ax.errorbar(energybins, data_t, xerr=xerror, **kwargs)
    
    return ax


def plot_energy_spectra(nonproton_data, proton_data, electron_data):
    

    
    

    fig, axs = plt.subplots(4, 1, dpi=300, sharex=True, figsize=(6,8))
    
    axs[0] = plot_single_data(electron_data, t1, axs[0], color='r', marker='o')
    axs[0] = plot_single_data(electron_data, t2, axs[0], color='b', marker='o')
    
    axs[1] = plot_single_data(proton_data, t1, axs[1], color='r', marker='o')
    axs[1] = plot_single_data(proton_data, t2, axs[1], color='b', marker='o')
    
    axs[2] = plot_single_data(nonproton_data[1], t1, axs[2], color='r', marker='o')
    axs[2] = plot_single_data(nonproton_data[1], t2, axs[2], color='b', marker='o')
    
    axs[3] = plot_single_data(nonproton_data[2], t1, axs[3], color='r', marker='o')
    axs[3] = plot_single_data(nonproton_data[2], t2, axs[3], color='b', marker='o')
    
    axs[2] = plot_single_data(nonproton_data[0], t1, axs[2], 
                              color='r', marker='o', ls='dashed', alpha=0.5)
    axs[2] = plot_single_data(nonproton_data[0], t2, axs[2], 
                              color='b', marker='o', ls='dashed', alpha=0.5)
    
    axs[3] = plot_single_data(nonproton_data[0], t1, axs[3], 
                              color='r', marker='o', ls='dotted', alpha=0.5)
    axs[3] = plot_single_data(nonproton_data[0], t2, axs[3], 
                              color='b', marker='o', ls='dotted', alpha=0.5)
    
    axs[1].text(0.8, 0.8, 'a) e-', transform=axs[0].transAxes, weight='bold')
    axs[1].text(0.8, 0.8, 'b) H+', transform=axs[1].transAxes, weight='bold')
    axs[2].text(0.8, 0.8, 'c) O+', transform=axs[2].transAxes, weight='bold')
    axs[3].text(0.8, 0.8, 'd) S+', transform=axs[3].transAxes, weight='bold')
    
    for a in axs:
        a.label_outer()
        a.set_yscale('log')
        a.set_xscale('log')
        a.set_ylabel('Intensity', weight='bold')
        a.xaxis.set_major_locator(ticker.LogLocator(base=10, numticks=15))
        a.grid('on', which='both', lw=0.5, ls='dotted')
    
    axs[-1].set_xlabel('E (keV)', weight='bold')
    
    axs[0].text(1.0, 1.1, '{:s}'.format(t1[0]), 
                transform=axs[0].transAxes,
                color='r', horizontalalignment='right')
    
    axs[0].text(0.0, 1.1, '{:s}'.format(t2[0]), 
                transform=axs[0].transAxes,
                color='b', horizontalalignment='left')
    
    fig.savefig('out_images/energyspectra_{:s}.png'.format(t1[0]),
                facecolor='white', bbox_inches='tight')
    plt.show()
    
plot_energy_spectra(nonproton_data, proton_data, electron_data)

In [ ]:
def plot_single_pa_data(data, t, ax, **kwargs):
    
    xerror=10
    
    pa = data.pitchangles + 5
    
    data_t = data.pitchangletime[t[0]: t[1]].values.T
    
    kwargs['marker'] = None
    ax.step(pa, data_t, where='mid', **kwargs)
    
    return ax


def plot_pa_spectra(nonproton_data, proton_data, electron_data):
    
#     t1 = ['2017-03-17 06:44:30', '2017-03-17 06:45:00']
#     t2 = ['2017-03-17 06:38:30', '2017-03-17 06:39:00']
    
    

    fig, axs = plt.subplots(4, 1, dpi=300, sharex=True, figsize=(6,8))
    
    axs[0] = plot_single_pa_data(electron_data, t1, axs[0], color='r', marker='o')
    axs[0] = plot_single_pa_data(electron_data, t2, axs[0], color='b', marker='o')
    
    axs[1] = plot_single_pa_data(proton_data, t1, axs[1], color='r', marker='o')
    axs[1] = plot_single_pa_data(proton_data, t2, axs[1], color='b', marker='o')
    
    axs[2] = plot_single_pa_data(nonproton_data[1], t1, axs[2], color='r', marker='o')
    axs[2] = plot_single_pa_data(nonproton_data[1], t2, axs[2], color='b', marker='o')
    
    axs[3] = plot_single_pa_data(nonproton_data[2], t1, axs[3], color='r', marker='o')
    axs[3] = plot_single_pa_data(nonproton_data[2], t2, axs[3], color='b', marker='o')
    
#     axs[2] = plot_single_pa_data(nonproton_data[0], t1, axs[2], 
#                                   color='r', marker='o', ls='dashed', alpha=0.5)
#     axs[2] = plot_single_pa_data(nonproton_data[0], t2, axs[2], 
#                                   color='b', marker='o', ls='dashed', alpha=0.5)
    
#     axs[3] = plot_single_pa_data(nonproton_data[0], t1, axs[3], 
#                                   color='r', marker='o', ls='dotted', alpha=0.5)
#     axs[3] = plot_single_pa_data(nonproton_data[0], t2, axs[3], 
#                                   color='b', marker='o', ls='dotted', alpha=0.5)
    
    kwargs = dict(weight='bold', bbox=dict(facecolor='w', edgecolor='k', pad=2.))
    
    axs[1].text(0.1, 0.1, 'a) e-', transform=axs[0].transAxes, **kwargs)
    axs[1].text(0.1, 0.1, 'b) H+', transform=axs[1].transAxes, **kwargs)
    axs[2].text(0.1, 0.1, 'c) O+', transform=axs[2].transAxes, **kwargs)
    axs[3].text(0.1, 0.1, 'd) S+', transform=axs[3].transAxes, **kwargs)
    
    for a in axs:
        a.label_outer()
        a.set_yscale('log')
        # a.set_xscale('log')
        a.set_ylabel('Diff. Intensity', weight='bold')
        # a.xaxis.set_major_locator(ticker.LogLocator(base=10, numticks=15))
        a.set_xticks(np.arange(0, 190, 30))
        a.set_xlim([0, 180])
        a.grid('on', which='both', lw=0.5, ls='dotted')
    
    axs[-1].set_xlabel('PA (deg)', weight='bold')
    
    axs[0].text(1.0, 1.1, '{:s}'.format(t1[0]), 
                transform=axs[0].transAxes,
                color='r', horizontalalignment='right')
    
    axs[0].text(0.0, 1.1, '{:s}'.format(t2[0]), 
                transform=axs[0].transAxes,
                color='b', horizontalalignment='left')
    
#     fig.savefig('out_images/PAspectra_{:s}.png'.format(t1[0]),
#                 facecolor='white', bbox_inches='tight')
    plt.show()
    
plot_pa_spectra(nonproton_data, proton_data, electron_data)